In [ ]:
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

from libs.dataset import anonymize_dataset, download_images
from libs.plotting import *
from libs.train import kfold_cross_validation
from libs.utils import find_root_repository

## Dataset

IF YOU NEVER DOWNLOAD LOCALY IMAGGES + ANONYMISE IT

In [ ]:
train_csv_path = str(find_root_repository() / "dataset/source/csv/sources_kfold_train.csv")
test_csv_path = str(find_root_repository()/ "dataset/source/csv/sources_kfold_test.csv")
images_ref_path = str(find_root_repository()/ "dataset/image_ref")

anonymize_dataset(train_csv_path)
download_images(train_csv_path, f"{images_ref_path}/train")

anonymize_dataset(test_csv_path)
download_images(test_csv_path, f"{images_ref_path}/test")

CODE FOR GENERATING KFOLD DATASET

In [ ]:
def split_into_fold(data: str, n: int) -> list:
    df = pd.read_csv(data)

    sgkf = StratifiedGroupKFold(
        n_splits=n,
        shuffle=True,
        random_state=42
    )
    
    folds = []

    for train_idx, val_idx in sgkf.split(
            X=df["SOURCE"],
            y=df["SIDE"],
            groups=df["PATIENT"]):
                
        train = df.iloc[train_idx]
        val = df.iloc[val_idx]
        
        folds.append({"train": train, "valid": val})
        
    return folds

def _create_fold_class(dict_ds: dict, dir: Path, class_name: str):
    
    output = dir / class_name
    df = dict_ds[class_name]
    
    left_path = output / "left"
    right_path = output / "right"
    
    left_path.mkdir(parents=True, exist_ok=True)
    right_path.mkdir(parents=True, exist_ok=True)
        
    left_df: pd.DataFrame = df.loc[df["SIDE"] == "L"]
    right_df: pd.DataFrame = df.loc[df["SIDE"] == "R"]

    root = find_root_repository()
    
    for _, row in left_df.iterrows():
        
        source = str(root / row["LOCAL_REF"])
        name = str(Path(row["LOCAL_REF"]).name)
        
        source = source.strip('"')
        name = name.strip()

        shutil.copy(source, left_path / name)

    for _, row in right_df.iterrows():
        
        source = str(root / row["LOCAL_REF"])
        name = str(Path(row["LOCAL_REF"]).name)
        
        source = source.strip('"')
        name = name.strip()

        shutil.copy(source, right_path / name)
        
def create_folds(lst_split: list, output_folder: str):
        
    output_path = Path(output_folder)
    
    for n_fold, dict_fold in enumerate(lst_split):
        
        fold_path = output_path / f"fold_{n_fold}"
                        
        # create for train
        _create_fold_class(dict_fold, fold_path, "train")
        
        # create for valid
        _create_fold_class(dict_fold, fold_path, "valid")

In [ ]:
dataset_csv_path = find_root_repository() / "dataset/source/csv/sources_kfold_train.csv"
dataset_path = find_root_repository() / "dataset/kfold"

list_folds = split_into_fold(str(dataset_csv_path), 5)
create_folds(list_folds, str(dataset_path))

## Training


In [ ]:
config = {"model": str(find_root_repository() / "models/yolo11n-cls.pt"),
            "optimizer": 'SGD',
            "batch": 16,
            "epochs": 100,
            "imgsz": 512}

os.environ["YOLO_VERBOSE"] = "False"
histories, metrics = kfold_cross_validation(5, "dataset/kfold", config)
metrics

In [ ]:
fig = plot_kfold_report(histories, metrics, ["accuracy", "precision", "rappel", "F1 score"])
add_text_to_report("model: yolo11x-cls.pt")
add_fig_to_report(fig)

## Brute Force the best config

In [ ]:
configs = {"model": ["models/yolo11x-cls.pt"], # models/yolo26s-cls.pt", "models/yolo26m-cls.pt", "models/yolo26x-cls.pt
          "optimizer": ['SGD', 'AdamW'], 
          "batch": [8, 16, 32],
          "epochs": [100],
          "imgsz": 512}

def find_best_config(report: str):
    for model in configs["model"]:
        for optimizer in configs["optimizer"]:
            for batch in configs["batch"]:
                for epochs in configs["epochs"]:
                    
                    config = {"model": model, 
                              "optimizer": optimizer,
                              "batch": batch,
                              "epochs": epochs,
                              "imgsz": 512}
                    
                    scores, metrics = kfold_cross_validation(5, "dataset/kfold", config)
                    scores = np.asarray(scores)
                    
                    
                    fig = plot_kfold_report(histories, metrics[0], metrics, ["scores", "accuracy", "rappel", "F1 score"],
                                            show=False)
                    
                    add_text_to_report(f"model: {model}, optimizer: {optimizer}, batch: {batch}, epochs: {epochs}", path=report)
                    add_fig_to_report(fig, path=report)                  

find_best_config("report_2.html");